# Issues
- For vertical levels, sigma-level approximation is used instead of the true hybrid level coordinate system. Pressure levels defined by sigma values are calculated as$$p_{\text{level}} = \sigma_{\text{level}} * p_{\text{surface}}$$ and these $\sigma_{\text{level}}$ are retrieved from [this FAQ page](https://rapidrefresh.noaa.gov/faq/HRRR.faq.html). However, I suspect that this source is outdated as StormCast documentation refers to HRRR's vertical coordinate system as **hybrid** levels and section 2.1 of [this documentation](https://opensky.ucar.edu/islandora/object/technotes%3A576) of Advanced Research WRF model v4 describes a hybrid level system that I suspect HRRR v4 (the latest version) also uses. Anyway I couldn't find a source for the parameters corresponding to each level in the hybrid system so the sigma levels are used as a first attempt approximation.

- HRRR uses Lambert Conformal Grid. However my conversion from ERA5's lat/lon grid to HRRR-format input just bilinearly interpolates the ERA5 grid (something like a spherical coordinate grid) to the required resolution. Again, suitable for a first approximation.

- In `MyLocalData.__call__()`, in the case that either `stored_times` (same as `self.times`) or `requested_times` has duplicates, `matches = np.where(stored_times == requested_time)[0]` will always pick the first match. Would be cleaner to just reject time arrays with duplicates in the first place. The same could probably be done for `variables`.

- Composite reflectivity, `refc`, is one of HRRR's 99 variables but does not exist in ERA5. Set to zero everywhere initial conditions for now.

- Some functions are designed to handle multiple initialisation times while others are designed to only handle one (ie assumes that `len(times)==1`). Just stick to one initialisation time for now – I don't quite get the purpose of multiple initialisation times yet.

- The main container for the data right now is a multi-dimensional numpy array, but manipulating this requires a lot of cross-checking with the correct indices for each variable. It would make more sense for `MyLocalData.__init__` to create `self.data` as a `xr.DataArray`. All Earth2Studio requires is for a `MyLocalData` object to be callable as `object(time=..., variable=...)` and to return an `xr.DataArray`.
  - On a similar note, there is some inconsistency in the input/output of the interpolation functions. For example, `hinterp` has input and output as a 2D numpy array, whereas `hinterp_all_levels` outputs a 3D numpy array but takes in an `xr.DataArray` as input.

In [1]:
# ── 0. Install & imports ──────────────────────────────────────────────────────

!pip install -q earth2studio[stormcast]

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import earth2studio.run as run

from datetime import datetime
from scipy.interpolate import RegularGridInterpolator
from earth2studio.models.px import StormCast
from earth2studio.data import GFS_FX
from earth2studio.io import ZarrBackend

In [2]:
# ── 1. Configuration ──────────────────────────────────────────────────────────

# File paths
PATH_SURFACE    = "/content/surface_variables.nc"
PATH_PRESSURE   = "/content/pressure_level_variables.nc"
PATH_SP         = "/content/surface_pressure.nc"
PATH_OUTPUT     = "stormcast_output_era5.zarr"

# ERA5 variable name → StormCast variable name (surface/single-level)
ERA5_SURFACE_MAP = {
    "t2m": "t2m",
    "u10": "u10m",
    "v10": "v10m",
    "msl": "mslp",
}

# ERA5 variable name → StormCast variable prefix (pressure-level)
ERA5_PRESSURE_MAP = {
    "u": "u",
    "v": "v",
    "t": "t",
    "q": "q",
    "z": "Z",   # ERA5 geopotential [m²/s²] → StormCast geopotential height [m]
}

# Mapping of HRRR hybrid level number → approximate sigma value
HRRR_SIGMA = {
     1: 1.0000,  2: 0.9980,  3: 0.9940,  4: 0.9870,  5: 0.9750,
     6: 0.9590,  7: 0.9390,  8: 0.9160,  9: 0.8920, 10: 0.8650,
    11: 0.8350, 13: 0.7660, 15: 0.6850, 20: 0.4565, 25: 0.3078,
    30: 0.2188,
}

# Variable units, used for colourbar labels
UNITS = {
    "t2m": "K",   "u10m": "m/s", "v10m": "m/s", "mslp": "Pa",
    "refc": "dBZ",
}

def infer_unit(var: str) -> str:
    if var in UNITS:
        return UNITS[var]
    prefix = var[0]
    return {"t": "K", "u": "m/s", "v": "m/s", "q": "kg/kg",
            "Z": "m", "p": "Pa"}.get(prefix, "")

In [3]:
# ── 2. Model & coordinate setup ───────────────────────────────────────────────

package = StormCast.load_default_package()
model   = StormCast.load_model(package, conditioning_data_source=GFS_FX())

coords    = model.input_coords()
VARIABLES = list(coords["variable"])
HRRR_Y    = np.asarray(coords["hrrr_y"])
HRRR_X    = np.asarray(coords["hrrr_x"])
NY, NX    = len(HRRR_Y), len(HRRR_X)

In [4]:
# ── 3. Helper functions ───────────────────────────────────────────────────────

class MyLocalData:
    """
    Takes in data as the input (which at this point is a 4D numpy array of unlabelled numbers)
    as well as the labels for each dimension. Also verifies that the length of each set of labels
    is equal to the actual data provided. Sets everything up to be easily accessible via self
    as per what Earth2Studio expects.

    Args:
        data:      float32 array of shape (nt, ny, nx, nc)
        variables: list of nc variable names matching StormCast's input coords
        hrrr_y:    1D array of length ny — Lambert Conformal y coordinates (m)
        hrrr_x:    1D array of length nx — Lambert Conformal x coordinates (m)
        times:     list of nt datetimes, one per time slice in data
    """

    def __init__(self, data, variables, hrrr_y, hrrr_x, times):
        nt, ny, nx, nc = data.shape
        assert len(times) == nt,     f"Expected {nt} timestamps, got {len(times)}"
        assert len(variables) == nc, f"Expected {nc} variables, got {len(variables)}"
        assert len(hrrr_y) == ny and len(hrrr_x) == nx

        self.data = data.astype(np.float32)
        self.variables = list(variables)
        self.hrrr_y, self.hrrr_x = hrrr_y, hrrr_x
        self.times = np.atleast_1d(np.array(times, dtype="datetime64[ns]"))

    def __call__(self, time, variable):
        """
        Takes in the requested time(s) and variable(s) and returns the slice of self.data
        in xr.DataArray format.

        The actual conversion from requested inputs to an xr.DataArray object only happens at the
        da = xr.DataArray(...) line, everything above that is just checking the validity of the input

        Note the transposition at the end. ERA5's default format is (t, y, x, vars)
        but Earth2Studio wants (t, vars, y, x).
        """

        # ensures that input variable is in array format
        if isinstance(variable, str):
            variable = [variable]

        # ensures that input time is in numpy array format,
        # regardless of whether a single time or an array was inputted
        requested_times = np.atleast_1d(np.array(time, dtype="datetime64[ns]"))
        stored_times = self.times

        t_indices = []

        for requested_time in requested_times:
            # in case there are duplicates in requested_times
            matches = np.where(stored_times == requested_time)[0]

            if len(matches) == 0:
                raise ValueError(
                    f"Requested time {requested_time} not found. "
                    f"Available times: {stored_times}"
                )

            t_indices.append(int(matches[0]))

        da = xr.DataArray(
            self.data[t_indices],
            dims=["time", "hrrr_y", "hrrr_x", "variable"],
            coords={
                "time": requested_times,
                "hrrr_y": self.hrrr_y,
                "hrrr_x": self.hrrr_x,
                "variable": self.variables,
            },
        )

        return da.sel(variable=variable).transpose(
            "time", "variable", "hrrr_y", "hrrr_x"
        )


def hinterp(field, src_lats, src_lons, target_lats, target_lons):
    """
    Using bilinear interpolation, take an input grid src_lats * src_lons and return the
    interpolated gric target_lats * target_lons

    Note that this is an approximation as HRRR uses lambert conformal coordinates while ERA5 uses lat/lon

    Input and output fields are 2D maps/grids/matrices
    """

    # ensures that latitudes are in ascending order
    # this is a requirement for RegularGridInterpolator
    if src_lats[0] > src_lats[-1]:
        src_lats = src_lats[::-1]
        field = field[::-1, :]

    # our interpolation function. if target point is outside original domain, returns NaN
    fn = RegularGridInterpolator(
        (src_lats, src_lons), field,
        method="linear", bounds_error=False, fill_value=np.nan,
    )

    # build target grid. lons2d/lats2d returns the lon/lat at the input grid pt [j, i]
    lons2d, lats2d = np.meshgrid(target_lons, target_lats)

    # RegularGridInterpolator wants points as a list, not as a grid
    # ie it wants [(x0, y0), (x0, y1), ... (x1, y0) ...]
    pts = np.stack([lats2d.ravel(), lons2d.ravel()], axis=-1)

    # interpolate and reshape back into 2D map of 32-bit floats
    return fn(pts).reshape(len(target_lats), len(target_lons)).astype(np.float32)


def hinterp_all_levels(
    da_pressure_var,
    src_lats,
    src_lons,
    target_lats,
    target_lons,
    time_index=0,
):
    """
    The xr.DataArray of pressure-level variables downloaded from ERA5 looks something
    like (variable, time, pressure level, latitute, longitude).

    Slice this dataset by variable (eg ds["z"], ds["t"], etc), and submit that slice
    into this function. This function will loop through all the pressure levels in
    ds["z"], ds["t"], etc and regrid (interpolate) each to the target grid.

    Note that although an xr.DataArray is inputted, the output is a numpy array

    Args:
        da_pressure_var:
            ERA5 DataArray with dimensions:
            (valid_time, pressure_level, latitude, longitude)

        src_lats, src_lons:
            Source ERA5 latitude/longitude coordinates.

        target_lats, target_lons:
            Target latitude/longitude coordinates.

        time_index:
            Which valid_time index to use. Defaults to 0.
            (just interpret time as valid_time. the meaning of valid time
             has some significance but it's not important in this context)

    Returns:
        NumPy array with shape:
            (n_pressure_levels, n_target_lats, n_target_lons)
    """
    regridded_levels = []

    n_pressure_levels = len(da_pressure_var.pressure_level)

    for pressure_index in range(n_pressure_levels):
        # Select one 2D pressure-level slice: (latitude, longitude)
        field_2d = da_pressure_var.isel(
            valid_time=time_index,
            pressure_level=pressure_index,
        ).values.astype(np.float32)

        # Regrid that 2D slice to the target grid: (target_lat, target_lon)
        field_2d_regridded = hinterp(
            field=field_2d,
            src_lats=src_lats,
            src_lons=src_lons,
            target_lats=target_lats,
            target_lons=target_lons,
        )

        regridded_levels.append(field_2d_regridded)
        # regridded_levels is a list of 2D grids

    # Stack all pressure levels into one 3D array:
    # (pressure_level, target_lat, target_lon)
    return np.stack(regridded_levels, axis=0)

def vinterp(field_3d, src_pressure_pa, target_pressure_pa):
    """
    Vertically interpolate a 3D pressure-level field to a 2D target pressure field.

    Args:
        field_3d:
            Array of shape (n_levels, NY, NX).
            Example: ERA5 temperature on pressure levels after horizontal regridding.

        src_pressure_pa:
            1D array of ERA5 pressure levels in Pa, shape (n_levels,).

        target_pressure_pa:
            2D array of target pressures in Pa, shape (NY, NX).
            Example: sigma_level * surface_pressure.

    Returns:
        2D array of interpolated values, shape (NY, NX).
    """

    # 1. Sort pressure levels from low pressure to high pressure.
    # np.searchsorted expects the coordinate array to be sorted ascending.
    sort_order = np.argsort(src_pressure_pa)

    pressure_sorted = src_pressure_pa[sort_order]
    field_sorted = field_3d[sort_order, :, :]

    # 2. For each grid cell, find where the target pressure fits
    # between the available ERA5 pressure levels.

    # searchsorted returns the index where target_pressure_pa should be inserted
    # into the sorted array pressure_sorted to keep it sorted.
    # ie this is literally the index of the pressure level directly above the target pressure
    upper_index = np.searchsorted(pressure_sorted, target_pressure_pa)

    # Prevent indices from going outside the available pressure-level range.
    # forces 1 <= upper_index < len(pressure_sorted)
    # upper_index and lower_index are 2D grids of levels of shape (NY, NX)
    upper_index = np.clip(upper_index, 1, len(pressure_sorted) - 1)
    lower_index = upper_index - 1

    # 3. Build y/x index arrays so we can pick values column-by-column.
    y_indices = np.arange(target_pressure_pa.shape[0])[:, None]
    x_indices = np.arange(target_pressure_pa.shape[1])[None, :]

    # 4. Get the two pressure levels surrounding each target pressure.
    pressure_lower = pressure_sorted[lower_index]
    pressure_upper = pressure_sorted[upper_index]

    # 5. Get the field values at those two surrounding pressure levels.
    field_lower = field_sorted[lower_index, y_indices, x_indices]
    field_upper = field_sorted[upper_index, y_indices, x_indices]

    # 6. Compute linear interpolation weight.
    weight = (target_pressure_pa - pressure_lower) / (
        pressure_upper - pressure_lower
    )

    # 7. Linearly interpolate.
    interpolated = field_lower + weight * (field_upper - field_lower)

    return interpolated.astype(np.float32)

In [5]:
# ── 4. Build input array ──────────────────────────────────────────────────────

def build_input_array():
    """Construct the full (1, NY, NX, 99) input array from ERA5 files."""

    ds_surface   = xr.open_dataset(PATH_SURFACE)
    ds_pressure  = xr.open_dataset(PATH_PRESSURE)
    ds_sp        = xr.open_dataset(PATH_SP)

    src_lats     = ds_surface.latitude.values
    src_lons     = ds_surface.longitude.values
    target_lats  = np.linspace(src_lats.max(), src_lats.min(), NY)  # north → south
    target_lons  = np.linspace(src_lons.min(), src_lons.max(), NX)

    data = np.zeros((1, NY, NX, len(VARIABLES)), dtype=np.float32)

    # ── 4a. Surface variables (direct mapping) ────────────────────────────────
    for era5_name, sc_name in ERA5_SURFACE_MAP.items():
        raw = ds_surface[era5_name].isel(valid_time=0).values.astype(np.float32) # hard-coded to select the first (and only?) time
        field = hinterp(raw, src_lats, src_lons, target_lats, target_lons)
        if np.isnan(field).any():
            raise ValueError(f"NaNs after regridding {era5_name}")
        data[0, :, :, VARIABLES.index(sc_name)] = field
        print(f"  {era5_name:4s} → {sc_name:5s} | min={field.min():.2f}  max={field.max():.2f}")

    # ── 4b. Surface pressure (needed to compute hybrid level pressures) ───────
    sp_raw = ds_sp["sp"].isel(valid_time=0).values.astype(np.float32)
    sp     = hinterp(sp_raw, src_lats, src_lons, target_lats, target_lons)
    print(f"\n  sp          | min={sp.min():.1f}  max={sp.max():.1f}  mean={sp.mean():.1f} Pa")

    # ── 4c. Pressure-level variables (horizontal regrid then vertical interp) ──────
    era5_p_pa = ds_pressure.pressure_level.values * 100.0   # hPa → Pa

    # Regrid every ERA5 field to the target horizontal grid (done once per variable)
    pressure_3d = {
        name: hinterp_all_levels(ds_pressure[name], src_lats, src_lons, target_lats, target_lons)
        for name in ERA5_PRESSURE_MAP
    }

    for level, sigma in HRRR_SIGMA.items():
        p_target = (sigma * sp).astype(np.float32)      # (NY, NX) target pressure

        # Store the pressure field itself
        p_name = f"p{level}hl"
        if p_name in VARIABLES:
            data[0, :, :, VARIABLES.index(p_name)] = p_target

        # Interpolate each atmospheric variable to this hybrid level
        for era5_name, sc_prefix in ERA5_PRESSURE_MAP.items():
            sc_name = f"{sc_prefix}{level}hl"
            if sc_name not in VARIABLES:
                continue

            field = vinterp(pressure_3d[era5_name], era5_p_pa, p_target)

            if era5_name == "z":
                field = field / 9.80665    # geopotential [m²/s²] → height [m]

            data[0, :, :, VARIABLES.index(sc_name)] = field

    # refc has no ERA5 equivalent — left as zero (placeholder)

    return data, target_lats, target_lons, np.datetime64(ds_surface.valid_time.values[0], "ns")


print("Building input array...")
data, target_lats, target_lons, STARTING_TIME = build_input_array()

filled = [v for v in VARIABLES if np.any(data[0, :, :, VARIABLES.index(v)] != 0)]
print(f"\nFilled {len(filled)}/99 variables ({[v for v in VARIABLES if v not in filled]} left as zero)")

Building input array...
  t2m  → t2m   | min=288.05  max=301.83
  u10  → u10m  | min=-8.33  max=8.06
  v10  → v10m  | min=-6.74  max=5.00
  msl  → mslp  | min=100767.06  max=101442.03

  sp          | min=82697.3  max=101928.3  mean=100136.0 Pa

Filled 98/99 variables (['refc'] left as zero)


In [6]:
# ── 5. Wrap and sanity-check ──────────────────────────────────────────────────

my_data = MyLocalData(
    data=data, variables=VARIABLES,
    hrrr_y=HRRR_Y, hrrr_x=HRRR_X,
    times=[STARTING_TIME],
)

sample = my_data([STARTING_TIME], VARIABLES)
assert sample.dims == ("time", "variable", "hrrr_y", "hrrr_x")
assert sample.shape == (1, 99, NY, NX)
print(f"Input array verified: {sample.dims} {sample.shape}")

Input array verified: ('time', 'variable', 'hrrr_y', 'hrrr_x') (1, 99, 512, 640)


In [7]:
# ── 6. Inference ──────────────────────────────────────────────────────────────

io = ZarrBackend(PATH_OUTPUT, backend_kwargs={"overwrite": True})
io = run.deterministic(time=[STARTING_TIME], nsteps=2, prognostic=model, data=my_data, io=io)

2026-06-03 02:44:07.160 | INFO     | earth2studio.run:deterministic:78 - Running simple workflow!
2026-06-03 02:44:07.160 | INFO     | earth2studio.run:deterministic:85 - Inference device: cuda
2026-06-03 02:44:07.889 | SUCCESS  | earth2studio.run:deterministic:109 - Fetched data from MyLocalData
2026-06-03 02:44:08.183 | INFO     | earth2studio.run:deterministic:139 - Inference starting!



Fetching GFS data:   0%|          | 0/26 [00:00<?, ?it/s]

Fetching GFS data:   0%|          | 0/26 [00:00<?, ?it/s]

Fetching GFS data:   0%|          | 0/26 [00:00<?, ?it/s]

Fetching GFS data:   0%|          | 0/26 [00:00<?, ?it/s]

Fetching GFS data:   0%|          | 0/26 [00:00<?, ?it/s]

Fetching GFS data:   0%|          | 0/26 [00:00<?, ?it/s]

Fetching GFS data:   0%|          | 0/26 [00:00<?, ?it/s]

Fetching GFS data:   0%|          | 0/26 [00:00<?, ?it/s]

Fetching GFS data:   0%|          | 0/26 [00:00<?, ?it/s]

Fetching GFS data:   0%|          | 0/26 [00:00<?, ?it/s]

Fetching GFS data:   0%|          | 0/26 [00:00<?, ?it/s]

Fetching GFS data:   0%|          | 0/26 [00:00<?, ?it/s]

Fetching GFS data:   0%|          | 0/26 [00:00<?, ?it/s]

Fetching GFS data:   0%|          | 0/26 [00:00<?, ?it/s]

Fetching GFS data:   0%|          | 0/26 [00:00<?, ?it/s]

Fetching GFS data:   0%|          | 0/26 [00:00<?, ?it/s]

Fetching GFS data:   0%|          | 0/26 [00:00<?, ?it/

2026-06-03 02:44:11.615 | DEBUG    | earth2studio.data.gfs:fetch_array:367 - Fetching GFS grib file: noaa-gfs-bdp-pds/gfs.20250325/00/atmos/gfs.t00z.pgrb2.0p25.f000 429695463-1184094
2026-06-03 02:44:11.628 | DEBUG    | earth2studio.data.gfs:fetch_array:367 - Fetching GFS grib file: noaa-gfs-bdp-pds/gfs.20250325/00/atmos/gfs.t00z.pgrb2.0p25.f000 215251168-602774
2026-06-03 02:44:11.638 | DEBUG    | earth2studio.data.gfs:fetch_array:367 - Fetching GFS grib file: noaa-gfs-bdp-pds/gfs.20250325/00/atmos/gfs.t00z.pgrb2.0p25.f000 341886577-1238163
2026-06-03 02:44:11.648 | DEBUG    | earth2studio.data.gfs:fetch_array:367 - Fetching GFS grib file: noaa-gfs-bdp-pds/gfs.20250325/00/atmos/gfs.t00z.pgrb2.0p25.f000 267790775-926949
2026-06-03 02:44:11.656 | DEBUG    | earth2studio.data.gfs:fetch_array:367 - Fetching GFS grib file: noaa-gfs-bdp-pds/gfs.20250325/00/atmos/gfs.t00z.pgrb2.0p25.f000 407574009-960606
2026-06-03 02:44:11.666 | DEBUG    | earth2studio.data.gfs:fetch_array:367 - Fetching GF

Fetching GFS data:   0%|          | 0/26 [00:00<?, ?it/s]

Fetching GFS data:   0%|          | 0/26 [00:00<?, ?it/s]

Fetching GFS data:   0%|          | 0/26 [00:00<?, ?it/s]

Fetching GFS data:   0%|          | 0/26 [00:00<?, ?it/s]

Fetching GFS data:   0%|          | 0/26 [00:00<?, ?it/s]

2026-06-03 02:44:11.828 | DEBUG    | earth2studio.data.gfs:fetch_array:367 - Fetching GFS grib file: noaa-gfs-bdp-pds/gfs.20250325/00/atmos/gfs.t00z.pgrb2.0p25.f000 346446268-951942
2026-06-03 02:44:11.837 | DEBUG    | earth2studio.data.gfs:fetch_array:367 - Fetching GFS grib file: noaa-gfs-bdp-pds/gfs.20250325/00/atmos/gfs.t00z.pgrb2.0p25.f000 209370589-752563
2026-06-03 02:44:11.845 | DEBUG    | earth2studio.data.gfs:fetch_array:367 - Fetching GFS grib file: noaa-gfs-bdp-pds/gfs.20250325/00/atmos/gfs.t00z.pgrb2.0p25.f000 268717724-931803
2026-06-03 02:44:11.853 | DEBUG    | earth2studio.data.gfs:fetch_array:367 - Fetching GFS grib file: noaa-gfs-bdp-pds/gfs.20250325/00/atmos/gfs.t00z.pgrb2.0p25.f000 399092975-1216410


...

2026-06-03 02:45:19.865 | DEBUG    | earth2studio.data.gfs:fetch_array:367 - Fetching GFS grib file: noaa-gfs-bdp-pds/gfs.20250325/00/atmos/gfs.t00z.pgrb2.0p25.f001 427042490-965397
2026-06-03 02:45:19.879 | DEBUG    | earth2studio.data.gfs:fetch_array:367 - Fetching GFS grib file: noaa-gfs-bdp-pds/gfs.20250325/00/atmos/gfs.t00z.pgrb2.0p25.f001 213428944-1169341
2026-06-03 02:45:19.890 | DEBUG    | earth2studio.data.gfs:fetch_array:367 - Fetching GFS grib file: noaa-gfs-bdp-pds/gfs.20250325/00/atmos/gfs.t00z.pgrb2.0p25.f001 341468135-903401
2026-06-03 02:45:19.905 | DEBUG    | earth2studio.data.gfs:fetch_array:367 - Fetching GFS grib file: noaa-gfs-bdp-pds/gfs.20250325/00/atmos/gfs.t00z.pgrb2.0p25.f001 216861627-594939
2026-06-03 02:45:19.915 | DEBUG    | earth2studio.data.gfs:fetch_array:367 - Fetching GFS grib file: noaa-gfs-bdp-pds/gfs.20250325/00/atmos/gfs.t00z.pgrb2.0p25.f001 267068264-1267148
2026-06-03 02:45:19.923 | DEBUG    | earth2studio.data.gfs:fetch_array:367 - Fetching GF


Fetching GFS data:   0%|          | 0/26 [00:00<?, ?it/s]

Fetching GFS data:   0%|          | 0/26 [00:00<?, ?it/s]

Fetching GFS data:   0%|          | 0/26 [00:00<?, ?it/s]

Fetching GFS data:   0%|          | 0/26 [00:00<?, ?it/s]

Fetching GFS data:   0%|          | 0/26 [00:00<?, ?it/s]

Fetching GFS data:   0%|          | 0/26 [00:00<?, ?it/s]

Fetching GFS data:   0%|          | 0/26 [00:00<?, ?it/s]

Fetching GFS data:   0%|          | 0/26 [00:00<?, ?it/s]

Fetching GFS data:   0%|          | 0/26 [00:00<?, ?it/s]

2026-06-03 02:45:20.069 | DEBUG    | earth2studio.data.gfs:fetch_array:367 - Fetching GFS grib file: noaa-gfs-bdp-pds/gfs.20250325/00/atmos/gfs.t00z.pgrb2.0p25.f001 406189544-958512
2026-06-03 02:45:20.081 | DEBUG    | earth2studio.data.gfs:fetch_array:367 - Fetching GFS grib file: noaa-gfs-bdp-pds/gfs.20250325/00/atmos/gfs.t00z.pgrb2.0p25.f001 428007887-945783
2026-06-03 02:45:20.090 | DEBUG    | earth2studio.data.gfs:fetch_array:367 - Fetching GFS grib file: noaa-gfs-bdp-pds/gfs.20250325/00/atmos/gfs.t00z.pgrb2.0p25.f001 342371536-846761
2026-06-03 02:45:20.098 | DEBUG    | earth2studio.data.gfs:fetch_array:367 - Fetching GFS grib file: noaa-gfs-bdp-pds/gfs.20250325/00/atmos/gfs.t00z.pgrb2.0p25.f001 421730155-875186
2026-06-03 02:45:20.110 | DEBUG    | earth2studio.data.gfs:fetch_array:367 - Fetching GFS grib file: noaa-gfs-bdp-pds/gfs.20250325/00/atmos/gfs.t00z.pgrb2.0p25.f001 400232447-860246
2026-06-03 02:45:20.121 | DEBUG    | earth2studio.data.gfs:fetch_array:367 - Fetching GFS 

Fetching GFS data: 100%|██████████| 26/26 [00:03<00:00,  8.35it/s]

Running inference: 100%|██████████| 3/3 [02:20<00:00, 46.85s/it]

2026-06-03 02:46:28.741 | SUCCESS  | earth2studio.run:deterministic:151 - 
Inference complete


In [8]:
# ── 7. Plot helpers ───────────────────────────────────────────────────────────

def plot_fields(fields_dict, title_prefix="", ncols=4):
    """Plot a dict of {var_name: 2D array} in a grid."""
    names = list(fields_dict)
    nrows = int(np.ceil(len(names) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows), squeeze=False)
    for ax, name in zip(axes.ravel(), names):
        im = ax.imshow(fields_dict[name], origin="upper")
        ax.set_title(f"{title_prefix}{name}")
        ax.set_xlabel("hrrr_x")
        ax.set_ylabel("hrrr_y")
        plt.colorbar(im, ax=ax, label=infer_unit(name))
    for ax in axes.ravel()[len(names):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()


def plot_vertical_profile(data, variables, j=NY // 2, i=NX // 2):
    """Plot vertical profiles of each variable group at a single grid cell."""
    for prefix, label in [("t", "K"), ("q", "kg/kg"), ("u", "m/s"),
                           ("v", "m/s"), ("Z", "m"), ("p", "Pa")]:
        levels, values = [], []
        for lev in HRRR_SIGMA:
            name = f"{prefix}{lev}hl"
            if name in variables:
                levels.append(lev)
                values.append(data[0, j, i, variables.index(name)])
        if not values:
            continue
        plt.figure(figsize=(5, 6))
        plt.plot(values, levels, marker="o")
        plt.gca().invert_yaxis()
        plt.xlabel(f"{prefix}#hl  ({label})")
        plt.ylabel("StormCast hybrid level")
        plt.title(f"Vertical profile at grid centre: {prefix}#hl")
        plt.grid(True)
        plt.show()

In [9]:
# ── 8. Plots ──────────────────────────────────────────────────────────────────

# Input fields
INPUT_VARS = ["t2m", "u10m", "v10m", "mslp",
              "t5hl", "q5hl", "u5hl", "Z5hl", "p5hl",
              "t20hl", "q20hl", "u20hl", "Z20hl",
              "t30hl", "u30hl", "Z30hl", "refc"]

plot_fields(
    {v: data[0, :, :, VARIABLES.index(v)] for v in INPUT_VARS},
    title_prefix="Input: ",
)

plot_vertical_profile(data, VARIABLES)

# Output fields
ds_out     = xr.open_zarr(PATH_OUTPUT)
OUTPUT_VARS = ["t2m", "mslp", "t5hl", "q5hl", "Z5hl", "t20hl", "Z20hl", "refc"]
n_leads    = ds_out.sizes["lead_time"]
n_vars     = len(OUTPUT_VARS)

fig, axes = plt.subplots(n_vars, n_leads, figsize=(5 * n_leads, 4 * n_vars), squeeze=False)
for row, var in enumerate(OUTPUT_VARS):
    for col in range(n_leads):
        field = ds_out[var].isel(time=0, lead_time=col).values
        im = axes[row, col].imshow(field, origin="upper")
        axes[row, col].set_title(f"{var} — lead {col}h")
        axes[row, col].set_xlabel("hrrr_x")
        axes[row, col].set_ylabel("hrrr_y")
        plt.colorbar(im, ax=axes[row, col], label=infer_unit(var))
plt.tight_layout()
plt.show()

Output hidden; open in https://colab.research.google.com to view.